# FisheriesAudit ALG 2026 — Entrega #05
## Modelo predictivo de sobreasignación de cuotas pesqueras

**Autor:** Ariel L. Giamportone  
**Filiación:** Ingeniero Pesquero | Docente Investigador | Data Scientist  
**Serie:** FisheriesAudit ALG 2026 — Gobernanza Pesquera Argentina  
**Fecha:** 2026-05-31

---

### Resumen

Se presenta el desarrollo y evaluación de un modelo de clasificación supervisada para predecir si el Consejo Federal Pesquero (CFP) aprobará una Captura Máxima Permisible (CMP) superior a la Captura Biológicamente Aceptable (CBA) recomendada por el INIDEP — evento denominado *sobreasignación*. Se evalúan Random Forest y Regresión Logística con validación cruzada estratificada. Las variables predictoras incluyen el estado del stock (FAO FIRMS), la magnitud de la CBA recomendada, capturas históricas y características temporales.

> ⚠️ **Nota metodológica:** El target (CMP/CBA > 1) es **sintético** en esta demostración, calibrado sobre datos reales de la literatura (Bertolotti et al. 2001; tasa histórica ~65% de sobreasignación). El pipeline completo (400+ actas CFP procesadas) generará el target real a partir de resoluciones de sesión.

**Palabras clave:** machine learning, gobernanza pesquera, Random Forest, SHAP, clasificación binaria, CFP Argentina, FisheriesAudit ALG

---

### Hipótesis

> **H1:** Es posible predecir con AUC-ROC > 0.65 si el CFP aprobará CMP > CBA, a partir de características del stock y contexto de la decisión.

> **H2:** El estado del stock (FAO: sobrexplotado vs. plena explotación) es la variable más predictiva de sobreasignación.

> **H3:** El año de la decisión tiene efecto positivo en la probabilidad de sobreasignación (tendencia histórica creciente).


In [ ]:
%matplotlib inline
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path(".").resolve()))

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from src.analysis.inidep_comparator import INIDEPComparator
from src.acquisition.fao_firms_scraper import FAOFIRMSScraper
from src.analysis.research_exporter import ModelExporter, SERIES_BRAND
from src.analysis.linkedin_formatter import LinkedInPost, HASHTAGS_PERSONAL, HASHTAGS_PESQUEROS_IA, SERIE_HEADER

DB_PATH = Path("data/processed/catalog.db")
OUT_DIR = Path("outputs/FisheriesAudit_ALG")
OUT_DIR.mkdir(parents=True, exist_ok=True)

comp = INIDEPComparator(DB_PATH)
comp.compute_comparisons()

fao = FAOFIRMSScraper(DB_PATH)
fao.seed_data()

mexp = ModelExporter(comp, fao, OUT_DIR)

print("✓ ModelExporter inicializado")
print(f"  sklearn + SHAP disponibles")
print(f"  Output: {OUT_DIR}")


## 1. Diseño del modelo

### 1.1 Variable objetivo

$$y_i = \mathbf{1}[\text{CMP}_i > \text{CBA}_i]$$

Donde:
- **CMP** = Captura Máxima Permisible aprobada por el CFP en sesión
- **CBA** = Captura Biológicamente Aceptable recomendada por el INIDEP

Un valor $y_i = 1$ indica *sobreasignación*: el CFP aprueba más del límite científico.

### 1.2 Variables predictoras

| Feature | Fuente | Descripción |
|---------|--------|-------------|
| **CBA INIDEP** | INIDEP ITOs | Recomendación científica en miles de toneladas |
| **Año** | CFP Actas | Año de la sesión — captura tendencia histórica |
| **Especie** | INIDEP/CFP | Categoría codificada numéricamente |
| **Estado stock** | FAO FIRMS | Ordinal: sub-explotado=0 → agotado=4 |
| **Captura/CBA año previo** | SAGPyA/SIPA | Presión de captura histórica |
| **CBA alternativa / CBA** | INIDEP ITOs | Margen de cautela científica |
| *(con corpus completo)* | CFP Actas | Quórum, unanimidad, HHI empresarial, red empresas |

### 1.3 Algoritmos evaluados

- **Random Forest** (200 árboles, max_depth=4, class_weight=balanced)
- **Regresión Logística** (con StandardScaler, class_weight=balanced)
- **Evaluación**: AUC-ROC en CV estratificada 5-fold, Precisión, Recall, F1


In [ ]:
# Construir matriz de features
X, y, feature_names = mexp.build_feature_matrix(synthetic_seed=42)

print(f"Dataset: {len(X)} observaciones, {len(feature_names)} features")
print(f"Sobreasignación (y=1): {y.sum()} ({y.mean():.1%})")
print(f"Sin sobreasignación (y=0): {(y==0).sum()} ({(y==0).mean():.1%})")
print()
print("Features:")
for i, fn in enumerate(feature_names):
    col = X.columns[i]
    print(f"  [{i}] {fn:35s} media={X[col].mean():.3f}, std={X[col].std():.3f}")
print()
print("Primeras 8 filas:")
X.head(8)


## 2. Análisis exploratorio de features

Distribución de las variables predictoras por clase (sobreasignación vs. sin sobreasignación).


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.flatten()

colores = {0: "#2196F3", 1: "#FF5722"}
labels = {0: "CMP ≤ CBA (sin sobreasig.)", 1: "CMP > CBA (sobreasig.)"}

for i, (col, fname) in enumerate(zip(X.columns, feature_names)):
    ax = axes[i]
    for clase in [0, 1]:
        subset = X[col][y == clase]
        ax.hist(subset, bins=12, alpha=0.6, color=colores[clase],
                label=labels[clase], density=True)
    ax.set_title(fname, fontsize=9)
    ax.set_ylabel("Densidad", fontsize=8)
    if i == 0:
        ax.legend(fontsize=7)

fig.suptitle("Distribución de features por clase — sobreasignación CFP", fontsize=11)
fig.text(0.99, 0.01, SERIES_BRAND, ha="right", va="bottom", fontsize=7, color="gray")
fig.tight_layout()
plt.savefig(OUT_DIR / "figuras" / "eda_features.png", dpi=300, bbox_inches="tight")
plt.show()
print("Figura guardada: eda_features.png")


## 3. Entrenamiento y evaluación

In [ ]:
result = mexp.train_and_evaluate(X, y, feature_names)

print("=== Resultados de evaluación ===")
print()
print(f"Random Forest")
print(f"  AUC-ROC CV (5-fold) : {result['rf_auc_cv']:.3f} ± {result['rf_scores'].std():.3f}")
print(f"  AUC-ROC por fold    : {[f'{s:.3f}' for s in result['rf_scores']]}")
print()
print(f"Regresión Logística")
print(f"  AUC-ROC CV (5-fold) : {result['lr_auc_cv']:.3f} ± {result['lr_scores'].std():.3f}")
print()

from sklearn.metrics import classification_report
print("=== Informe de clasificación (Random Forest, datos de entrenamiento) ===")
print(classification_report(result['y'], result['rf'].predict(result['X_clean']),
                             target_names=["CMP ≤ CBA", "CMP > CBA"]))


## 4. Curvas ROC

**Figura.** Curvas ROC de Random Forest y Regresión Logística.
AUC-ROC > 0.5 indica capacidad predictiva; AUC = 1.0 es predicción perfecta.


In [ ]:
fig = mexp.figura_roc_curve(result, save=True)
plt.show()
print(f"RF AUC = {result['rf_auc_cv']:.3f} | LR AUC = {result['lr_auc_cv']:.3f}")


## 5. Importancia de variables

**Figura.** Importancia de cada feature según la disminución media del índice Gini (Random Forest).
Permite identificar qué características del contexto de decisión son más predictivas de sobreasignación.


In [ ]:
fig = mexp.figura_feature_importance(result, save=True)
plt.show()

importances = result['rf'].feature_importances_
print("=== Importancias de features (Gini) ===")
for fname, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    bar = "█" * int(imp * 40)
    print(f"  {fname:35s} {imp:.4f}  {bar}")


## 6. Valores SHAP — interpretabilidad por observación

SHAP (SHapley Additive exPlanations) descompone la predicción de cada observación en contribuciones individuales de cada feature. Permite explicar decisiones individuales del modelo — una capacidad esencial para publicaciones científicas y uso en política pública.

Un valor SHAP positivo aumenta la probabilidad predicha de sobreasignación; negativo la reduce.


In [ ]:
fig = mexp.figura_shap_summary(result, save=True)
plt.show()
print("SHAP summary guardado en:", OUT_DIR / "figuras" / "shap_summary.png")


## 7. Matriz de confusión y calibración

In [ ]:
fig_cm = mexp.figura_confusion_matrix(result, save=True)
plt.show()

fig_cal = mexp.figura_calibracion(result, save=True)
plt.show()


## 8. Análisis de hipótesis

### H1: AUC-ROC > 0.65
### H2: Estado del stock como predictor dominante
### H3: Efecto temporal (año)


In [ ]:
importances = result['rf'].feature_importances_
fname_to_imp = dict(zip(feature_names, importances))

rf_auc = result['rf_auc_cv']
imp_stock = fname_to_imp.get("Estado stock (ord.)", 0)
imp_year = fname_to_imp.get("Año", 0)
imp_max = max(importances)
feature_max = feature_names[list(importances).index(imp_max)]

print("=== Evaluación de hipótesis ===")
print()
print(f"H1: AUC-ROC > 0.65")
print(f"    AUC-ROC RF (CV) = {rf_auc:.3f}")
print(f"    {'✅ CONFIRMADA' if rf_auc > 0.65 else '❌ NO CONFIRMADA'}")
print()
print(f"H2: Estado del stock como predictor dominante")
print(f"    Importancia 'Estado stock': {imp_stock:.4f}")
print(f"    Feature más importante: '{feature_max}' ({imp_max:.4f})")
print(f"    {'✅ CONFIRMADA' if feature_max == 'Estado stock (ord.)' else f'⚠️  Predictor dominante: {feature_max}'}")
print()
print(f"H3: Efecto temporal positivo (año)")
print(f"    Importancia 'Año': {imp_year:.4f}")
print(f"    (SHAP values positivos en años recientes → ver figura SHAP)")
print()
print("=== Nota metodológica ===")
print("Con datos sintéticos el AUC refleja la consistencia del patrón generativo,")
print("no el poder predictivo sobre datos reales futuros. Con el corpus completo")
print("de actas CFP las métricas representarán el rendimiento real del modelo.")


## 9. Hallazgo estructurado y exportación

In [ ]:
hallazgo = mexp.generar_hallazgo_modelo(result)
print(hallazgo.resumen_ejecutivo())
print()
print(f"Nivel de evidencia: {hallazgo.nivel_evidencia.upper()}")


In [ ]:
csv_path = mexp.exportar_predicciones_csv(result, X)
print(f"CSV predicciones: {csv_path}")

latex = mexp.exportar_latex_metricas(result)
print()
print("Tabla LaTeX métricas:")
print(latex[:500])


## 10. Posts LinkedIn — Entrega #05

In [ ]:
rf_auc = result['rf_auc_cv']
importances = result['rf'].feature_importances_
top_feature = feature_names[list(importances).index(max(importances))]

post_personal = LinkedInPost(
    numero_entrega=5,
    titulo="¿Puede la IA predecir cuándo el CFP ignora a la ciencia?",
    emoji_tema="🤖",
    hook=(
        "Entrenamos un modelo de machine learning con 25 años de datos pesqueros argentinos.\n"
        f"AUC-ROC = {rf_auc:.2f}. La variable más predictiva: '{top_feature}'."
    ),
    contexto=(
        "Si el CFP aprueba cuotas superiores a la recomendación científica del INIDEP "
        "en ~65% de los casos históricos, ¿hay un patrón? ¿Podemos predecirlo?\n\n"
        "FisheriesAudit ALG — Entrega #05 presenta el primer modelo predictivo "
        "de sobreasignación de cuotas pesqueras en Argentina. "
        "Features: estado del stock (FAO), CBA recomendada, capturas históricas, año."
    ),
    datos_principales=[
        f"AUC-ROC Random Forest (CV 5-fold): {rf_auc:.3f}",
        f"Variable más predictiva: {top_feature}",
        "Algoritmos: Random Forest + Regresión Logística + SHAP",
        "⚠️ Dataset sintético en esta versión — corpus completo en desarrollo",
        "Framework replicable en cualquier sistema de gestión pesquera",
    ],
    reflexion=(
        "Un modelo predictivo no reemplaza el análisis institucional. "
        "Lo que hace es señalar: 'en estas condiciones, históricamente el CFP "
        "aprobó más del límite científico'. "
        "Eso es suficiente para abrir una conversación de política pública."
    ),
    cta="¿Investigás gobernanza de recursos naturales? El código es abierto.",
    hashtags=HASHTAGS_PERSONAL,
    perfil="personal",
    fuentes=["INIDEP ITOs", "FAO FIRMS", "SAGPyA/SIPA", "CFP Actas Públicas"],
)

print("=== POST PERSONAL — Entrega #05 ===")
print(post_personal.render())


In [ ]:
post_ia = LinkedInPost(
    numero_entrega=5,
    titulo="Clasificación binaria aplicada a regulación pesquera: metodología",
    emoji_tema="⚙️",
    hook=(
        "¿Qué variables predicen si un organismo regulador ignorará la ciencia?\n"
        "Random Forest + SHAP sobre actas del Consejo Federal Pesquero argentino."
    ),
    contexto=(
        "Pipeline técnico de la Entrega #05 — FisheriesAudit ALG:\n"
        "• Target: y = 1 si CMP > CBA (sobreasignación)\n"
        "• Features: CBA INIDEP, estado stock FAO (ordinal), capturas SAGPyA (lag-1),\n"
        "  CBA alternativa/CBA ratio, año, especie encoded\n"
        "• Modelo: RandomForest(n_estimators=200, max_depth=4, class_weight='balanced')\n"
        "• Evaluación: StratifiedKFold(n_splits=5), AUC-ROC\n"
        "• Interpretabilidad: SHAP TreeExplainer + beeswarm personalizado matplotlib"
    ),
    datos_principales=[
        f"AUC-ROC RF (CV 5-fold): {rf_auc:.3f}",
        "class_weight='balanced' — maneja desbalance de clases",
        "SHAP TreeExplainer — interpretabilidad por observación",
        "Curva de calibración incluida — crítica para uso en política pública",
        "Framework extensible: +quórum, +votación, +HHI empresarial con corpus completo",
    ],
    reflexion=(
        "El AUC con datos sintéticos no es el resultado final — es la validación del pipeline. "
        "Con el corpus real (400+ actas procesadas) el modelo tendrá poder predictivo real. "
        "El valor está en el framework: replicable, auditable, open source."
    ),
    cta="¿Aplicás ML a regulación o política pública? Conversemos sobre la metodología.",
    hashtags=HASHTAGS_PESQUEROS_IA,
    perfil="pesqueros_ia",
    fuentes=["FisheriesAudit ALG", "scikit-learn", "SHAP", "CFP Actas Públicas"],
)

print("=== PESQUEROS EN IA — Entrega #05 ===")
print(post_ia.render())


## 11. Roadmap: modelo con corpus completo

### Features adicionales disponibles con pipeline completo

| Feature | Fuente | Impacto esperado |
|---------|--------|-----------------|
| `quorum_sesion` | CFP Actas | Quórum mínimo → mayor presión por aprobar |
| `unanimidad` | CFP Actas | Voto unánime → posible acuerdo previo |
| `hhi_especie` | Red NER | Alta concentración → mayor presión privada |
| `n_empresas_vinculadas` | Grafo | Más actores → más lobby |
| `ratio_previo_cmp_cba` | CFP histórico | Lag del target — tendencia inercial |
| `mes_sesion` | CFP Actas | Estacionalidad (veda/pesca) |
| `mandato_gobierno` | Externo | Período presidencial |

### Mejoras metodológicas planificadas

1. **XGBoost** como alternativa al Random Forest
2. **SMOTE** para sobremuestreo de la clase minoritaria
3. **Permutation importance** además de Gini
4. **SHAP interaction values** — pares de features
5. **Análisis por especie** — modelos individuales por especie con suficientes datos

### Comando para generar corpus completo

```bash
python scripts/run_full_pipeline.py --step download --years 1998-2025
python scripts/run_full_pipeline.py --step process
python scripts/run_full_pipeline.py --step audit --limit 500
# Luego re-ejecutar este notebook
```

---

### Metodología

- **Target sintético:** calibrado sobre tasa histórica Argentina (~65% sobreasignación, Bertolotti et al. 2001; Lerena 2009)
- **Validación:** CV estratificada para preservar distribución de clases en cada fold
- **Interpretabilidad:** SHAP permite explicar decisiones individuales — requisito ético para uso en política pública

### Declaración

El análisis es descriptivo-metodológico y no constituye acusación legal.  
Código abierto: github.com/arielgiamportone/cfp-audit-intelligence

---

*FisheriesAudit ALG 2026 — Ariel L. Giamportone*  
*Ing. Pesquero | Docente Investigador | Data Scientist*
